# Spectral Analysis — Phase 4: Multi-Dataset, Multi-Model Generalization

**Datasets**: MATH-500 (competition math, 300 samples) · GPQA Diamond (graduate science MCQ, 198 samples)  
**Models**: Qwen2.5-Math-1.5B · Qwen2.5-Math-7B · DeepSeek-Math-7B · DeepSeek-R1-Distill-Llama-8B · Llama-3.1-8B · Qwen2.5-7B  
**Temperature**: T=1.5 (better class balance, amplifies entropy dynamics)  
**Features**: 12 signals — all Phase 3 signals + trace_length  
**Builds on**: Phase 3 best — sw_var_peak 72.9–73.5% standalone, 75.0–75.9% fusion

### Usage
Run all cells top to bottom. The pipeline loop (Cell 6) processes every model-dataset config automatically.
Results are saved per config — interrupt and restart safely, completed configs are skipped.

### Pipeline
| Config | Model | Dataset | Samples |
|--------|-------|---------|--------|
| A1 | Qwen2.5-Math-1.5B-Instruct | MATH-500 | 300 |
| A2 | Qwen2.5-Math-7B-Instruct | MATH-500 | 300 |
| A3 | DeepSeek-Math-7B-Instruct | MATH-500 | 300 |
| A4 | DeepSeek-R1-Distill-Llama-8B | MATH-500 | 300 |
| B1 | Llama-3.1-8B-Instruct | GPQA Diamond | 198 |
| B2 | Qwen2.5-7B-Instruct | GPQA Diamond | 198 |
| B3 | DeepSeek-R1-Distill-Llama-8B | GPQA Diamond | 198 |

### Decision gates
| Gate | Criterion |
|------|----------|
| G1 | sw_var_peak AUC > 71.8% on ≥ 4 of 6 model-dataset configs |
| G2 | Best fusion > best single signal on ≥ 5 of 6 configs |
| G3 | Best fusion spread ≤ 5pp within each dataset group |

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'accelerate', 'datasets', 'scipy'], check=True)

# Authenticate with HuggingFace (needed for gated models like Llama-3.1-8B)
try:
    from huggingface_hub import login
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace login OK.')
except Exception as e:
    print(f'HF login skipped: {e}')

print('Ready.')

Mounted at /content/drive
HuggingFace login OK.
Ready.


In [2]:
import os

BASE_DIR = '/content/drive/MyDrive/epr_spectral_phase4'
os.makedirs(BASE_DIR, exist_ok=True)

# ── Pipeline definition — do not edit below this line ─────────────────────────
PIPELINE = [
    # Experiment A: MATH-500, math-specialized models
    {'tag': 'A1', 'model_id': 'Qwen/Qwen2.5-Math-1.5B-Instruct',        'dataset': 'math500', 'n_samples': 300, 'temp': 1.5, 'max_new': 1024},
    {'tag': 'A2', 'model_id': 'Qwen/Qwen2.5-Math-7B-Instruct',          'dataset': 'math500', 'n_samples': 300, 'temp': 1.5, 'max_new': 1024},
    {'tag': 'A3', 'model_id': 'deepseek-ai/deepseek-math-7b-instruct',   'dataset': 'math500', 'n_samples': 300, 'temp': 1.5, 'max_new': 1024},
    {'tag': 'A4', 'model_id': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B','dataset': 'math500', 'n_samples': 300, 'temp': 1.5, 'max_new': 1536},
    # Experiment B: GPQA Diamond, general instruction models
    {'tag': 'B1', 'model_id': 'mistralai/Mistral-7B-Instruct-v0.2',     'dataset': 'gpqa', 'n_samples': 198, 'temp': 1.5,    'max_new': 768},
    {'tag': 'B2', 'model_id': 'Qwen/Qwen2.5-7B-Instruct',               'dataset': 'gpqa',    'n_samples': 198, 'temp': 1.5, 'max_new': 768},
    {'tag': 'B3', 'model_id': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B','dataset': 'gpqa',    'n_samples': 198, 'temp': 1.5, 'max_new': 768},
    {'tag': 'B4', 'model_id': 'meta-llama/Llama-3.1-8B-Instruct',        'dataset': 'gpqa',    'n_samples': 198, 'temp': 1.5, 'max_new': 768},
]

for cfg in PIPELINE:
    short = cfg['model_id'].split('/')[-1]
    cfg['model_short'] = short
    cfg['run_key']     = f"{short}__{cfg['dataset']}"
    cfg['run_dir']     = os.path.join(BASE_DIR, cfg['run_key'])
    os.makedirs(cfg['run_dir'], exist_ok=True)

print(f'Pipeline: {len(PIPELINE)} runs')
for cfg in PIPELINE:
    done_path = os.path.join(cfg['run_dir'], 'phase4_results.pkl')
    status = 'DONE' if os.path.exists(done_path) else 'pending'
    print(f"  [{cfg['tag']}] {cfg['model_short']:40s} {cfg['dataset']:8s} {status}")

Pipeline: 8 runs
  [A1] Qwen2.5-Math-1.5B-Instruct               math500  DONE
  [A2] Qwen2.5-Math-7B-Instruct                 math500  DONE
  [A3] deepseek-math-7b-instruct                math500  DONE
  [A4] DeepSeek-R1-Distill-Llama-8B             math500  DONE
  [B1] Mistral-7B-Instruct-v0.2                 gpqa     DONE
  [B2] Qwen2.5-7B-Instruct                      gpqa     pending
  [B3] DeepSeek-R1-Distill-Llama-8B             gpqa     pending
  [B4] Llama-3.1-8B-Instruct                    gpqa     pending


In [3]:
import numpy as np, pickle, re, itertools, gc, torch, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from scipy.linalg import eigh
from scipy.signal import stft as scipy_stft
from tqdm import tqdm

def load_cache(path):
    return pickle.load(open(path, 'rb')) if os.path.exists(path) else {}

def save_cache(obj, path):
    pickle.dump(obj, open(path, 'wb'))

def free_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def load_model(model_id):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, device_map='auto', torch_dtype=torch.float16, trust_remote_code=True)
    mdl.eval(); print(f'Loaded {model_id}'); return mdl, tok

def fmt_prompt(tok, msg):
    try: return tok.apply_chat_template([{'role':'user','content':msg}],
                                         tokenize=False, add_generation_prompt=True)
    except: return f'<|user|>\n{msg}\n<|assistant|>\n'

def token_entropies_from_scores(scores, K=15):
    ents = []
    for s in scores:
        lp   = F.log_softmax(s[0], dim=-1)
        topk = torch.topk(lp, min(K, lp.shape[-1])).values
        p    = torch.exp(topk); p = p / (p.sum() + 1e-12)
        ents.append(-(p * torch.log(p + 1e-12)).sum().item())
    return ents

def generate_full(mdl, tok, prompt_msg, temperature=1.5, K=15, max_new_tokens=1024):
    prompt = fmt_prompt(tok, prompt_msg)
    inputs = tok(prompt, return_tensors='pt').to(mdl.device)
    if 'token_type_ids' in inputs: del inputs['token_type_ids']
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens,
                           do_sample=True, temperature=temperature, top_k=50,
                           output_scores=True, return_dict_in_generate=True,
                           pad_token_id=tok.eos_token_id)
    gen_ids   = out.sequences[0][inputs.input_ids.shape[1]:]
    full_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    all_ents  = token_entropies_from_scores(out.scores, K)
    return full_text, all_ents

def boot_auc(y, scores, n=1000):
    y, s = np.array(y), np.array(scores)
    if len(np.unique(y)) < 2 or np.std(s) < 1e-8: return float('nan'), float('nan'), float('nan')
    base = roc_auc_score(y, s)
    rng  = np.random.default_rng(42); boots = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        boots.append(roc_auc_score(y[idx], s[idx]))
    lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (base, base)
    return base, lo, hi

def nadler_fuse(*views):
    X = np.column_stack(views); n, k = X.shape
    C = np.cov(X.T)
    if C.ndim == 0: C = np.array([[float(C)]])
    try:
        off = C - np.diag(np.diag(C))
        rs, cs = off.sum(1), off.sum(0)
        M = np.diag(rs) @ np.linalg.pinv(C) @ np.diag(cs)
        _, vecs = eigh(M)
        w = np.abs(vecs[:, -1]); w /= w.sum() + 1e-12
    except Exception: w = np.ones(k) / k
    return X @ w, w

print('Core helpers defined.')

Core helpers defined.


In [4]:
def compute_spectral_features(ents, min_len=8):
    e = np.array(ents, dtype=float); N = len(e)
    if N < min_len: return None
    e_ac     = e - e.mean()
    fft_vals = np.fft.rfft(e_ac)
    psd      = np.abs(fft_vals) ** 2
    freqs    = np.fft.rfftfreq(N)
    psd_sum  = psd.sum() + 1e-12
    psd_norm = psd / psd_sum
    spec_ent = -np.sum(psd_norm * np.log(psd_norm + 1e-12))
    low_mask  = (freqs > 0.0) & (freqs <= 0.10)
    high_mask = (freqs >= 0.40) & (freqs <= 0.50)
    low_power  = psd[low_mask].sum()  / psd_sum
    high_power = psd[high_mask].sum() / psd_sum
    hl_ratio   = high_power / (low_power + 1e-12)
    ac_mask    = freqs > 0
    dom_freq   = float(freqs[ac_mask][np.argmax(psd[ac_mask])]) if ac_mask.sum() > 0 else 0.0
    centroid   = float(np.sum(freqs[ac_mask] * psd_norm[ac_mask]) /
                       (psd_norm[ac_mask].sum() + 1e-12)) if ac_mask.sum() > 0 else 0.0
    return {'spectral_entropy': float(spec_ent), 'low_band_power': float(low_power),
            'high_band_power': float(high_power), 'hl_ratio': float(hl_ratio),
            'dominant_freq': dom_freq, 'spectral_centroid': centroid}

def compute_stft_features(ents, nperseg=16, noverlap=8, min_len=32):
    e = np.array(ents, dtype=float)
    if len(e) < min_len:
        return {'stft_max_high_power': 0.0, 'stft_spectral_entropy': 0.0}
    e_ac = e - e.mean()
    f, t, Zxx = scipy_stft(e_ac, nperseg=nperseg, noverlap=noverlap)
    psd = np.abs(Zxx) ** 2
    high_mask = f >= 0.40
    if high_mask.sum() > 0 and psd.shape[1] > 0:
        high_frac = psd[high_mask].sum(0) / (psd.sum(0) + 1e-12)
        max_local_high = float(high_frac.max())
    else:
        max_local_high = 0.0
    psd_n = psd / (psd.sum(0, keepdims=True) + 1e-12)
    frame_ent = -np.sum(psd_n * np.log(psd_n + 1e-12), axis=0)
    stft_ent = float(frame_ent.mean()) if len(frame_ent) > 0 else 0.0
    return {'stft_max_high_power': max_local_high, 'stft_spectral_entropy': stft_ent}

def compute_time_domain(ents, tail_frac=0.20, sw_window=16, sw_step=8):
    e = np.array(ents, dtype=float)
    W = max(1, int(len(e) * tail_frac))
    rpdi = float(e[-W:].mean() / (e.mean() + 1e-12))
    if len(e) >= sw_window:
        sw_vars = [np.var(e[i:i+sw_window]) for i in range(0, len(e)-sw_window+1, sw_step)]
        sw_var_peak = float(np.max(sw_vars))
    else:
        sw_var_peak = float(np.var(e))
    return {'rpdi': rpdi, 'sw_var_peak': sw_var_peak}

def extract_all_features(ents):
    e = np.array(ents, dtype=float)
    result = {'epr': float(e.mean()), 'trace_length': float(len(e))}
    gf = compute_spectral_features(ents)
    if gf is None: return None
    result.update(gf)
    result.update(compute_stft_features(ents))
    result.update(compute_time_domain(ents))
    return result

FEAT_NAMES = ['epr', 'trace_length', 'spectral_entropy', 'low_band_power', 'high_band_power',
              'hl_ratio', 'dominant_freq', 'spectral_centroid',
              'stft_max_high_power', 'stft_spectral_entropy', 'rpdi', 'sw_var_peak']
print(f'Feature set ({len(FEAT_NAMES)} signals): {FEAT_NAMES}')

Feature set (12 signals): ['epr', 'trace_length', 'spectral_entropy', 'low_band_power', 'high_band_power', 'hl_ratio', 'dominant_freq', 'spectral_centroid', 'stft_max_high_power', 'stft_spectral_entropy', 'rpdi', 'sw_var_peak']


In [5]:
from datasets import load_dataset

# ── MATH-500 ──────────────────────────────────────────────────────────────────
def load_math500(n_samples=300):
    attempts = [
        ('lighteval/MATH_500',        {},                   'test'),
        ('HuggingFaceH4/MATH-500',    {},                   'test'),
        ('EleutherAI/hendrycks_math', {'name': 'all'},      'test'),
        ('EleutherAI/hendrycks_math', {'name': 'algebra'},  'test'),
    ]
    for path, kwargs, split in attempts:
        try:
            ds = load_dataset(path, split=split, **kwargs)
            samples = [ds[i] for i in range(min(n_samples, len(ds)))]
            print(f'Loaded {len(samples)} MATH problems from {path}.')
            return samples
        except Exception as e:
            print(f'  {path} failed: {e}')
    raise RuntimeError('Could not load MATH dataset from any source.')

def math_prompt(row):
    q = row.get('problem', row.get('query', row.get('question', '')))
    return (f'Solve the following competition math problem. '
            f'Show all your work step by step, then give your final answer in \\boxed{{}}.\n\n{q}')

def extract_math_answer(text):
    m = re.search(r'\\boxed\{([^}]*)\}', text)
    if m:
        val = re.sub(r'[^\d\.\-\/\(\)]', '', m.group(1).replace(',', ''))
        if val: return val
    nums = re.findall(r'[\-\d]+(?:\.\d+)?', text.replace(',', ''))
    return nums[-1] if nums else ''

def is_correct_math(gen, gold_row):
    sol = gold_row.get('solution', gold_row.get('answer', gold_row.get('output', '')))
    p, g = extract_math_answer(gen), extract_math_answer(sol)
    if not p or not g: return False
    try: return abs(float(p) - float(g)) < 1e-6
    except: return p.strip() == g.strip()

# ── GPQA Diamond ─────────────────────────────────────────────────────────────
def load_gpqa():
    ds = load_dataset('Idavidrein/gpqa', 'gpqa_diamond', split='train')
    print(f'Loaded {len(ds)} GPQA Diamond problems.')
    return list(ds)

def gpqa_prompt_and_answer(row, idx):
    rng = np.random.default_rng(idx)
    choices = [row['Correct Answer'],
               row['Incorrect Answer 1'],
               row['Incorrect Answer 2'],
               row['Incorrect Answer 3']]
    order   = rng.permutation(4)
    letters = ['A', 'B', 'C', 'D']
    shuffled = [choices[i] for i in order]
    correct_letter = letters[int(np.where(order == 0)[0][0])]
    opts = '\n'.join(f'{l}) {t}' for l, t in zip(letters, shuffled))
    prompt = (f'Answer the following graduate-level science question by selecting the best answer. '
              f'Think through your reasoning carefully.\n\n'
              f"{row['Question']}\n\n{opts}\n\n"
              f'Provide your reasoning, then state your final answer as a single letter: A, B, C, or D.')
    return prompt, correct_letter

def extract_gpqa_answer(text):
    patterns = [
        r'(?:answer is|answer:|the answer|final answer)[^A-Da-d]*([A-Da-d])\b',
        r'\b([A-Da-d])\s*(?:is correct|is the best|is the answer)',
        r'^\s*([A-Da-d])[\)\.]?\s*$',
    ]
    for p in patterns:
        matches = re.findall(p, text, re.IGNORECASE | re.MULTILINE)
        if matches: return matches[-1].upper()
    matches = re.findall(r'\b([A-D])\b', text.upper())
    return matches[-1] if matches else ''

def is_correct_gpqa(gen, correct_letter):
    return extract_gpqa_answer(gen) == correct_letter.upper()

print('Dataset loaders defined.')

Dataset loaders defined.


In [6]:
# ── Main inference pipeline — runs all configs automatically ──────────────────
math500_data, gpqa_data = None, None

for cfg in PIPELINE:
    cache_path   = os.path.join(cfg['run_dir'], 'inference_cache.pkl')
    results_path = os.path.join(cfg['run_dir'], 'phase4_results.pkl')

    if os.path.exists(results_path):
        print(f"[{cfg['tag']}] {cfg['run_key']} — already complete, skipping.")
        continue

    print(f"\n{'='*60}")
    print(f"[{cfg['tag']}] {cfg['model_short']} on {cfg['dataset']} (T={cfg['temp']})")
    print(f"{'='*60}")

    # Load dataset once per type
    if cfg['dataset'] == 'math500' and math500_data is None:
        math500_data = load_math500(300)
    if cfg['dataset'] == 'gpqa' and gpqa_data is None:
        gpqa_data = load_gpqa()

    dataset   = math500_data if cfg['dataset'] == 'math500' else gpqa_data
    n_samples = cfg['n_samples']

    # Load inference cache
    cache     = load_cache(cache_path)
    remaining = [i for i in range(n_samples) if not cache.get(i, {}).get('done')]
    print(f'Cache: {n_samples - len(remaining)}/{n_samples} done. Remaining: {len(remaining)}')

    if remaining:
        model, tokenizer = load_model(cfg['model_id'])
        for i in tqdm(remaining, desc=f"{cfg['tag']} inference"):
            try:
                row = dataset[i]
                if cfg['dataset'] == 'math500':
                    prompt   = math_prompt(row)
                    full_text, all_ents = generate_full(
                        model, tokenizer, prompt, cfg['temp'], max_new_tokens=cfg['max_new'])
                    correct  = is_correct_math(full_text, row)
                    gold_ref = row.get('solution', '')
                else:
                    prompt, correct_letter = gpqa_prompt_and_answer(row, i)
                    full_text, all_ents = generate_full(
                        model, tokenizer, prompt, cfg['temp'], max_new_tokens=cfg['max_new'])
                    correct  = is_correct_gpqa(full_text, correct_letter)
                    gold_ref = correct_letter
                cache[i] = {'done': True, 'full_text': full_text,
                            'all_entropies': all_ents, 'correct': correct, 'gold': gold_ref}
            except Exception as ex:
                print(f'  Error {i}: {ex}'); cache[i] = {'done': False}
            if i % 20 == 0: save_cache(cache, cache_path)
            free_memory()
        save_cache(cache, cache_path)
        del model, tokenizer; free_memory()

    # ── Feature extraction ────────────────────────────────────────────────────
    idx_ok    = sorted(i for i in range(n_samples)
                       if cache.get(i, {}).get('done') and cache[i].get('all_entropies'))
    labels, feat_list, n_toks = [], [], []
    for i in idx_ok:
        v = cache[i]
        ents = v['all_entropies']
        if len(ents) < 8: continue
        feats = extract_all_features(ents)
        if feats is None: continue
        labels.append(int(v['correct']))
        feat_list.append(feats)
        n_toks.append(len(ents))

    labels      = np.array(labels)
    n_toks      = np.array(n_toks)
    feat_arrays = {name: np.array([f[name] for f in feat_list]) for name in FEAT_NAMES}
    n_correct   = int(labels.sum())
    print(f'Samples: {len(labels)} | Correct: {n_correct} ({labels.mean():.1%}) | Avg trace: {n_toks.mean():.0f} tok')

    # ── Individual AUCs ───────────────────────────────────────────────────────
    auc_map, sign_map, rows = {}, {}, []
    for name in FEAT_NAMES:
        a_pos, lo_pos, hi_pos = boot_auc(labels,  feat_arrays[name])
        a_neg, lo_neg, hi_neg = boot_auc(labels, -feat_arrays[name])
        if a_pos >= a_neg: auc, lo, hi, sign = a_pos, lo_pos, hi_pos, +1
        else:              auc, lo, hi, sign = a_neg, lo_neg, hi_neg, -1
        auc_map[name] = auc; sign_map[name] = sign
        rows.append((auc, name, lo, hi))
    rows.sort(reverse=True)
    print(f'\nIndividual AUCs ({cfg["run_key"]})')
    for auc, name, lo, hi in rows:
        beat = ' **' if auc > 0.718 else ''
        print(f'  {name:<26} {100*auc:5.1f}%  [{100*lo:.1f}, {100*hi:.1f}]{beat}')

    # ── Pairwise rho ─────────────────────────────────────────────────────────
    oriented = {name: feat_arrays[name] * sign_map[name] for name in FEAT_NAMES}
    rho_mat  = {}
    for a, b in itertools.combinations_with_replacement(FEAT_NAMES, 2):
        rho, _ = spearmanr(oriented[a], oriented[b])
        rho_mat[(a, b)] = rho_mat[(b, a)] = rho

    # ── Combinatorial Nadler ──────────────────────────────────────────────────
    informative   = [n for n in FEAT_NAMES if auc_map[n] > 0.50]
    valid_subsets = []
    for size in range(2, len(informative) + 1):
        for subset in itertools.combinations(informative, size):
            if all(abs(rho_mat[(a,b)]) < 0.75 for a,b in itertools.combinations(subset, 2)):
                valid_subsets.append(subset)
    print(f'Valid subsets: {len(valid_subsets)}')

    subset_results = []
    for subset in tqdm(valid_subsets, desc='Nadler', leave=False):
        views = [oriented[n] for n in subset]
        fused, weights = nadler_fuse(*views)
        auc, lo, hi    = boot_auc(labels, fused)
        subset_results.append({'subset': subset, 'auc': auc, 'lo': lo, 'hi': hi,
                               'weights': {n: float(w) for n, w in zip(subset, weights)}})
    subset_results.sort(key=lambda x: -x['auc'])

    best = subset_results[0] if subset_results else None
    print(f'Best fusion: {"+".join(best["subset"])} = {100*best["auc"]:.1f}% [{100*best["lo"]:.1f}, {100*best["hi"]:.1f}]' if best else 'No valid subsets')
    print('Weights: ' + ', '.join(f'{k}={v:.3f}' for k,v in sorted(best['weights'].items(), key=lambda x:-x[1])) if best else '')

    # ── Save results ──────────────────────────────────────────────────────────
    result = {
        'tag': cfg['tag'], 'model': cfg['model_short'], 'dataset': cfg['dataset'],
        'n_samples': len(labels), 'accuracy': float(labels.mean()),
        'avg_trace': float(n_toks.mean()),
        'auc_map': auc_map, 'sign_map': sign_map, 'rho_mat': rho_mat,
        'subset_results': subset_results[:30], 'feat_names': FEAT_NAMES,
    }
    save_cache(result, results_path)
    print(f'Saved to {results_path}')

print('\nPipeline complete.')

[A1] Qwen2.5-Math-1.5B-Instruct__math500 — already complete, skipping.
[A2] Qwen2.5-Math-7B-Instruct__math500 — already complete, skipping.
[A3] deepseek-math-7b-instruct__math500 — already complete, skipping.
[A4] DeepSeek-R1-Distill-Llama-8B__math500 — already complete, skipping.
[B1] Mistral-7B-Instruct-v0.2__gpqa — already complete, skipping.

[B2] Qwen2.5-7B-Instruct on gpqa (T=1.5)


README.md: 0.00B [00:00, ?B/s]

gpqa_diamond.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

Loaded 198 GPQA Diamond problems.
Cache: 198/198 done. Remaining: 0
Samples: 198 | Correct: 60 (30.3%) | Avg trace: 571 tok

Individual AUCs (Qwen2.5-7B-Instruct__gpqa)
  spectral_centroid           58.3%  [49.7, 66.6]
  high_band_power             57.6%  [49.1, 65.9]
  hl_ratio                    57.4%  [48.7, 65.6]
  rpdi                        56.9%  [48.6, 65.7]
  low_band_power              56.8%  [48.1, 65.2]
  stft_spectral_entropy       56.3%  [48.1, 64.7]
  trace_length                54.8%  [46.1, 62.7]
  epr                         54.7%  [45.6, 63.4]
  dominant_freq               54.2%  [45.2, 62.5]
  stft_max_high_power         51.1%  [42.2, 59.4]
  sw_var_peak                 50.8%  [41.8, 59.6]
  spectral_entropy            50.4%  [41.7, 58.3]
Valid subsets: 1523


...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.1-8B-Instruct



B4 inference: 100%|██████████| 198/198 [1:19:54<00:00, 24.21s/it]


Samples: 198 | Correct: 53 (26.8%) | Avg trace: 593 tok

Individual AUCs (Llama-3.1-8B-Instruct__gpqa)
  hl_ratio                    55.6%  [45.8, 64.9]
  low_band_power              55.5%  [45.9, 64.9]
  spectral_centroid           55.5%  [46.3, 64.6]
  high_band_power             55.3%  [45.5, 64.9]
  stft_max_high_power         54.9%  [45.4, 63.7]
  stft_spectral_entropy       54.8%  [45.4, 64.6]
  trace_length                53.4%  [45.4, 62.0]
  dominant_freq               52.4%  [43.9, 61.4]
  spectral_entropy            51.7%  [42.2, 61.5]
  epr                         51.1%  [41.6, 60.2]
  sw_var_peak                 50.7%  [41.4, 60.1]
  rpdi                        50.7%  [41.4, 59.3]
Valid subsets: 563


Best fusion: stft_max_high_power+stft_spectral_entropy = 58.2% [48.8, 67.6]
Weights: stft_max_high_power=0.721, stft_spectral_entropy=0.279
Saved to /content/drive/MyDrive/epr_spectral_phase4/Llama-3.1-8B-Instruct__gpqa/phase4_results.pkl

Pipeline complete.


In [7]:
# ── Cross-model / cross-dataset comparison ────────────────────────────────────
all_results = {}
for cfg in PIPELINE:
    path = os.path.join(cfg['run_dir'], 'phase4_results.pkl')
    if os.path.exists(path):
        r = load_cache(path)
        all_results[cfg['tag']] = r
        print(f"[{cfg['tag']}] {cfg['run_key']} loaded")
    else:
        print(f"[{cfg['tag']}] {cfg['run_key']} missing")

if len(all_results) < 2:
    print('\nNeed at least 2 completed runs.')
else:
    tags_math = [t for t in all_results if all_results[t]['dataset'] == 'math500']
    tags_gpqa = [t for t in all_results if all_results[t]['dataset'] == 'gpqa']

    for group_name, tags in [('MATH-500', tags_math), ('GPQA Diamond', tags_gpqa)]:
        if not tags: continue
        print(f'\n{"="*80}')
        print(f'{group_name} — Individual signal AUCs')
        print(f'{"="*80}')
        col_w = 18
        print(f'{"Signal":<26}', end='')
        for t in tags:
            lbl = all_results[t]['model'][:16]
            print(f'{lbl:>{col_w}}', end='')
        print('  spread')
        print('-' * (26 + col_w * len(tags) + 8))
        ref_feats = all_results[tags[0]]['feat_names']
        ref_aucs  = all_results[tags[0]]['auc_map']
        for sig in sorted(ref_feats, key=lambda n: -ref_aucs.get(n, 0)):
            vals = [all_results[t]['auc_map'].get(sig, 0) for t in tags]
            print(f'{sig:<26}', end='')
            for v in vals: print(f'{100*v:{col_w-1}.1f}%', end='')
            spread = (max(vals) - min(vals)) * 100 if len(vals) > 1 else 0
            print(f'  {spread:.1f}pp')

        print(f'\n{"Best fusion":<26}', end='')
        best_aucs = []
        for t in tags:
            sr = all_results[t]['subset_results']
            b  = sr[0]['auc'] if sr else 0.0
            best_aucs.append(b)
            print(f'{100*b:{col_w-1}.1f}%', end='')
        spread = (max(best_aucs) - min(best_aucs)) * 100 if len(best_aucs) > 1 else 0
        print(f'  {spread:.1f}pp')

        print(f'{"Accuracy":<26}', end='')
        for t in tags: print(f'{100*all_results[t]["accuracy"]:{col_w-1}.1f}%', end='')
        print()
        print(f'{"Avg trace (tok)":<26}', end='')
        for t in tags: print(f'{all_results[t]["avg_trace"]:{col_w-1}.0f} ', end='')
        print()

        print(f'\nBest fusion per model ({group_name}):')
        for t in tags:
            sr = all_results[t]['subset_results']
            if sr:
                b = sr[0]
                print(f'  [{t}] {"+".join(b["subset"])} = {100*b["auc"]:.1f}% [{100*b["lo"]:.1f}, {100*b["hi"]:.1f}]')
                print('    w: ' + ', '.join(f'{k}={v:.3f}' for k,v in sorted(b['weights'].items(), key=lambda x:-x[1])))

    # Decision gates
    print(f'\n{"="*80}')
    print('DECISION GATES')
    print(f'{"="*80}')
    sw_beats = sum(1 for t in all_results if all_results[t]['auc_map'].get('sw_var_peak', 0) > 0.718)
    print(f'G1 (sw_var_peak > 71.8% on >= 4/6 configs): {"PASS" if sw_beats >= 4 else "FAIL"} ({sw_beats}/{len(all_results)} configs)')
    fusion_beats = sum(1 for t in all_results
                       if all_results[t]['subset_results'] and
                       all_results[t]['subset_results'][0]['auc'] > max(all_results[t]['auc_map'].values()))
    print(f'G2 (best fusion > best single on >= 5/6 configs): {"PASS" if fusion_beats >= 5 else "FAIL"} ({fusion_beats}/{len(all_results)} configs)')
    for group_name, tags in [('MATH-500', tags_math), ('GPQA Diamond', tags_gpqa)]:
        if len(tags) < 2: continue
        best_aucs = [all_results[t]['subset_results'][0]['auc'] for t in tags if all_results[t]['subset_results']]
        if best_aucs:
            spread = (max(best_aucs) - min(best_aucs)) * 100
            print(f'G3 ({group_name} spread <= 5pp): {"PASS" if spread <= 5.0 else "FAIL"} (spread={spread:.1f}pp)')

[A1] Qwen2.5-Math-1.5B-Instruct__math500 loaded
[A2] Qwen2.5-Math-7B-Instruct__math500 loaded
[A3] deepseek-math-7b-instruct__math500 loaded
[A4] DeepSeek-R1-Distill-Llama-8B__math500 loaded
[B1] Mistral-7B-Instruct-v0.2__gpqa loaded
[B2] Qwen2.5-7B-Instruct__gpqa loaded
[B3] DeepSeek-R1-Distill-Llama-8B__gpqa loaded
[B4] Llama-3.1-8B-Instruct__gpqa loaded

MATH-500 — Individual signal AUCs
Signal                      Qwen2.5-Math-1.5  Qwen2.5-Math-7B-  deepseek-math-7b  DeepSeek-R1-Dist  spread
----------------------------------------------------------------------------------------------------------
spectral_centroid                      86.6%             94.3%             65.2%             75.5%  29.2pp
low_band_power                         86.4%             94.1%             63.7%             75.1%  30.3pp
hl_ratio                               85.6%             94.3%             62.4%             74.1%  31.9pp
epr                                    85.6%             96.6%         